In [1]:
# NOTEBOOK 02c: PREPARE CLASSIFICATION LIST

import os
import re
from collections import Counter

import pandas as pd

QUALITY_PATH   = '../data/quality_check'
PROCESSED_PATH = '../data/processed'
os.makedirs(QUALITY_PATH, exist_ok=True)


In [2]:
# ----------------------------------------------------------
# STEP 0: LOAD DATA
# ----------------------------------------------------------

products = pd.read_csv(
    os.path.join(PROCESSED_PATH, 'products_cleaned.csv')
)

products['ingredients_list'] = products['ingredients_parsed'].apply(
    lambda x: [
        i.strip()
        for i in str(x).split('|')
        if i.strip() != '' and i.strip().lower() != 'nan'
    ]
    if pd.notna(x) else []
)

all_ingredients = [
    ing
    for sublist in products['ingredients_list']
    for ing in sublist
]
freq            = Counter(all_ingredients)
top_n           = 2000
top_ingredients = [ing for ing, _ in freq.most_common(top_n)]

print("STEP 0: Load Data")
print()
print(f"  {'Total unique ingredients':<40} {len(freq):>10,}")
print(f"  {'Working with top N ingredients':<40} {top_n:>10,}")
print(f"  {'Reconstructed ingredient mentions':<40} "
      f"{len(all_ingredients):>10,}")


STEP 0: Load Data

  Total unique ingredients                     13,899
  Working with top N ingredients                2,000
  Reconstructed ingredient mentions           261,561


In [3]:
# ----------------------------------------------------------
# STEP 1: REMOVE PARSING ARTIFACTS
# ----------------------------------------------------------

print()
print("STEP 1: Remove Parsing Artifacts")
print()

ARTIFACT_PATTERNS = [
    r'-$',
    r'^\d+\)$',
    r'^\d{5}\)$',
    r'^\d+\.?\d*\s*%',
    r'cotton wick',
    r'wick\.',
    r'paraben-$',
    r'phthalate-$',
    r'sulfate-$',
    r'silicone-$',
    r'fragrance-free',
    r'cruelty-free',
    r'^vegan$',
    r'gluten.free',
    r'free from',
    r'made from',
    r'manufactured by',
    r'^may contain',
    r'avoid contact',
    r'keep out of reach',
    r'^directions',
    r'^warning',
    r'broad spectrum',
    r'water resistant',
    r'spf \d+',
    r'inactive ingredient',
    r'active ingredient',
    r'other ingredient',
    r'dist\. by',
    r'mfg by',
    r'^and ',
    r'^or ',
    r'^\*+$',
    r'^-+$',
    r'^\++$',
]


def is_parsing_artifact(ingredient):
    if pd.isna(ingredient):
        return True
    name = str(ingredient).lower().strip()
    if not re.search(r'[a-z]', name):
        return True
    if len(name) > 100:
        return True
    for pattern in ARTIFACT_PATTERNS:
        if re.search(pattern, name):
            return True
    return False


original_count    = len(top_ingredients)
artifacts_removed = [
    ing for ing in top_ingredients
    if is_parsing_artifact(ing)
]
top_ingredients   = [
    ing for ing in top_ingredients
    if not is_parsing_artifact(ing)
]

pd.DataFrame([
    {'artifact': a, 'frequency': freq[a]}
    for a in artifacts_removed
]).to_csv(
    os.path.join(QUALITY_PATH, 'removed_parsing_artifacts.csv'),
    index=False
)

print(f"  {'Original top ingredients':<40} {original_count:>10,}")
print(f"  {'Artifacts removed':<40} {len(artifacts_removed):>10,}")
print(f"  {'Clean ingredients remaining':<40} "
      f"{len(top_ingredients):>10,}")



STEP 1: Remove Parsing Artifacts

  Original top ingredients                      2,000
  Artifacts removed                                 9
  Clean ingredients remaining                   1,991


In [4]:
# ----------------------------------------------------------
# STEP 2: NORMALIZE INGREDIENT NAME FORMATTING
# ----------------------------------------------------------

print()
print("STEP 2: Normalize Ingredient Name Formatting")
print()

SOY_EXCEPTION = 'glycine soja oil / soybean oil'


def normalize_ingredient_name(name):
    # Remove formatting artifacts: trailing *, % concentration, slash variants
    n = str(name).strip()
    if n.lower() == SOY_EXCEPTION.lower():
        return SOY_EXCEPTION
    n = n.replace('\\\\', '/').replace('\\', '/')
    n = n.replace(' / ', '/').replace(' /', '/').replace('/ ', '/')
    n = re.sub(r'\*+$', '', n).strip()
    n = re.sub(r'\s+\d+(?:\.\d+)?%$', '', n).strip()
    return n


original_names     = top_ingredients.copy()
top_ingredients    = [
    normalize_ingredient_name(ing)
    for ing in top_ingredients
]
formatting_cleaned = sum(
    1 for orig, norm in zip(original_names, top_ingredients)
    if orig != norm
)

print(f"  {'Ingredients with formatting cleaned':<40} "
      f"{formatting_cleaned:>10,}")



STEP 2: Normalize Ingredient Name Formatting

  Ingredients with formatting cleaned              79


In [5]:
# ----------------------------------------------------------
# STEP 3: APPLY MANUAL CANONICAL RULES
# ----------------------------------------------------------

print()
print("STEP 3: Apply Manual Canonical Rules")
print()

canonical_rules = [

    # WATER
    ('water',
     ['water', 'aqua', '/eau', 'eau/', 'eau)',
      'aqua(water', 'water (aqua', 'aqua / water',
      'water / aqua', 'water/aqua', 'aqua/water',
      'water\\aqua', 'aqua\\water', 'aqua (water',
      'water/aqua/eau', 'aqua/water/eau']),

    # ALCOHOLS
    ('alcohol denat',
     ['alcohol denat', 'denatured alcohol', 'sd alcohol',
      'alcohol (denat', 'denatured ethyl alcohol']),

    ('isopropyl alcohol',   ['isopropyl alcohol']),
    ('benzyl alcohol',      ['benzyl alcohol']),
    ('cetearyl alcohol',    ['cetearyl alcohol']),

    # FRAGRANCE AND ALLERGENS
    ('parfum (fragrance)',
     ['parfum', 'fragrance', 'perfume']),

    ('linalool',            ['linalool']),
    ('limonene',            ['limonene', 'd-limonene']),
    ('citronellol',         ['citronellol']),
    ('geraniol',            ['geraniol']),
    ('citral',              ['citral']),
    ('benzyl salicylate',   ['benzyl salicylate']),
    ('benzyl benzoate',     ['benzyl benzoate', 'benzl benzoate']),
    ('coumarin',            ['coumarin']),
    ('hexyl cinnamal',      ['hexyl cinnamal']),
    ('eugenol',             ['eugenol']),
    ('isoeugenol',          ['isoeugenol']),
    ('farnesol',            ['farnesol']),
    ('hydroxycitronellal',  ['hydroxycitronellal']),
    ('amyl cinnamal',       ['amyl cinnamal']),
    ('cinnamyl alcohol',    ['cinnamyl alcohol']),
    ('cinnamal',            ['cinnamal']),

    ('alpha-isomethyl ionone',
     ['alpha-isomethyl ionone', 'alphaisomethyl ionone']),

    ('butylphenyl methylpropional',
     ['butylphenyl methylpropional', 'lilial']),

    # EMOLLIENTS AND HUMECTANTS
    ('glycerin',
     ['glycerin', 'glycerol', 'glycerine']),

    ('caprylyl glycol',             ['caprylyl glycol']),
    ('butylene glycol',             ['butylene glycol']),
    ('propanediol',                 ['propanediol']),
    ('pentylene glycol',            ['pentylene glycol']),
    ('propylene glycol',            ['propylene glycol']),
    ('hexylene glycol',             ['hexylene glycol']),

    ('caprylic/capric triglyceride',
     ['caprylic/capric triglyceride',
      'caprylic capric triglyceride']),

    ('squalane',            ['squalane']),
    ('shea butter',         ['shea butter', 'butyrospermum parkii']),
    ('jojoba oil',          ['jojoba', 'simmondsia chinensis']),
    ('aloe barbadensis',    ['aloe barbadensis', 'aloe vera']),

    # PRESERVATIVES
    ('phenoxyethanol',          ['phenoxyethanol']),
    ('ethylhexylglycerin',      ['ethylhexylglycerin']),
    ('potassium sorbate',       ['potassium sorbate']),
    ('sodium benzoate',         ['sodium benzoate']),
    ('methylparaben',           ['methylparaben']),
    ('ethylparaben',            ['ethylparaben']),
    ('propylparaben',           ['propylparaben']),
    ('butylparaben',            ['butylparaben']),
    ('isobutylparaben',         ['isobutylparaben']),
    ('dmdm hydantoin',          ['dmdm hydantoin']),
    ('imidazolidinyl urea',     ['imidazolidinyl urea']),
    ('diazolidinyl urea',       ['diazolidinyl urea']),
    ('quaternium-15',           ['quaternium-15']),
    ('chlorphenesin',           ['chlorphenesin']),
    ('sodium dehydroacetate',   ['sodium dehydroacetate']),
    ('dehydroacetic acid',      ['dehydroacetic acid']),

    # VITAMINS AND ANTIOXIDANTS
    ('tocopherol',
     ['tocopherol', 'vitamin e']),

    ('tocopheryl acetate',  ['tocopheryl acetate']),

    ('ascorbic acid',
     ['ascorbic acid', 'vitamin c', 'ascorbyl']),

    ('retinol',             ['retinol', 'vitamin a']),
    ('niacinamide',         ['niacinamide', 'nicotinamide']),
    ('panthenol',           ['panthenol', 'provitamin b5']),
    ('caffeine',            ['caffeine']),
    ('bht',                 ['bht', 'butylated hydroxytoluene']),

    # THICKENERS AND STABILIZERS
    ('xanthan gum',         ['xanthan gum']),
    ('carbomer',            ['carbomer']),
    ('dimethicone',         ['dimethicone']),
    ('silica',              ['silica']),

    # pH ADJUSTERS AND CHELATORS
    ('citric acid',         ['citric acid']),
    ('sodium hydroxide',    ['sodium hydroxide']),

    ('disodium edta',
     ['disodium edta', 'tetrasodium edta', 'edta']),

    # MINERALS
    ('talc',                ['talc']),
    ('kaolin',              ['kaolin']),

    # UV FILTERS
    ('oxybenzone',
     ['oxybenzone', 'benzophenone-3']),

    ('octinoxate',
     ['octinoxate', 'ethylhexyl methoxycinnamate']),

    ('ethylhexyl salicylate',
     ['ethylhexyl salicylate', 'octyl salicylate']),

    ('butyl methoxydibenzoylmethane',
     ['butyl methoxydibenzoylmethane', 'avobenzone']),

    ('homosalate',          ['homosalate']),

    # SURFACTANTS
    ('sodium lauryl sulfate',   ['sodium lauryl sulfate']),
    ('sodium laureth sulfate',  ['sodium laureth sulfate']),
    ('cocamidopropyl betaine',  ['cocamidopropyl betaine']),

    # ACTIVE INGREDIENTS
    ('salicylic acid',      ['salicylic acid']),

    ('sodium hyaluronate',
     ['sodium hyaluronate', 'hyaluronic acid', 'hyaluronate']),

    ('hydrolyzed collagen',
     ['collagen', 'hydrolyzed collagen']),

    # PLANT EXTRACTS AND OILS
    ('helianthus annuus seed oil',
     ['helianthus annuus', 'sunflower seed oil']),

    ('cocos nucifera oil',
     ['cocos nucifera', 'coconut oil']),

    ('persea gratissima oil',
     ['persea gratissima', 'avocado oil']),

    ('argania spinosa kernel oil',
     ['argania spinosa', 'argan oil']),

    ('rosa canina fruit oil',
     ['rosa canina', 'rosehip']),

    ('camellia sinensis leaf extract',
     ['camellia sinensis', 'green tea']),

    ('rosmarinus officinalis extract',
     ['rosmarinus officinalis', 'rosemary']),

    # CI COLORANTS
    ('iron oxides (ci 77491)',
     ['ci 77491', 'iron oxide red', 'red iron oxide',
      'iron oxide (ci 77491)', 'iron oxides (ci 77491',
      'iron oxides ci 77491']),

    ('iron oxides (ci 77492)',
     ['ci 77492', 'iron oxide yellow', 'yellow iron oxide',
      'iron oxide (ci 77492)', 'iron oxides (ci 77492',
      'iron oxides ci 77492']),

    ('iron oxides (ci 77499)',
     ['ci 77499', 'iron oxide black', 'black iron oxide',
      'iron oxide (ci 77499)', 'iron oxides (ci 77499',
      'iron oxides ci 77499']),

    ('titanium dioxide (ci 77891)',
     ['titanium dioxide', 'ci 77891']),

    ('mica (ci 77019)',
     ['mica', 'ci 77019']),

    ('zinc oxide (ci 77947)',
     ['zinc oxide', 'ci 77947']),

    ('tin oxide (ci 77861)',
     ['tin oxide', 'ci 77861']),

    ('bismuth oxychloride (ci 77163)',
     ['bismuth oxychloride', 'ci 77163']),

    ('ultramarines (ci 77007)',
     ['ultramarines', 'ultramarine blue', 'ci 77007']),

    ('carmine (ci 75470)',
     ['carmine', 'cochineal', 'ci 75470', 'natural red 4']),

    ('ferric ferrocyanide (ci 77510)',
     ['ferric ferrocyanide', 'prussian blue', 'ci 77510']),

    ('manganese violet (ci 77742)',
     ['manganese violet', 'ci 77742']),

    ('ci 19140 (yellow 5)',
     ['ci 19140', 'yellow 5', 'fd&c yellow no. 5',
      'fd&c yellow #5', 'tartrazine']),

    ('ci 15985 (yellow 6)',
     ['ci 15985', 'yellow 6', 'fd&c yellow no. 6',
      'fd&c yellow #6', 'sunset yellow']),

    ('ci 42090 (blue 1)',
     ['ci 42090', 'blue 1', 'fd&c blue no. 1',
      'fd&c blue #1', 'brilliant blue']),

    ('ci 16035 (red 40)',
     ['ci 16035', 'red 40', 'fd&c red no. 40',
      'fd&c red #40', 'allura red']),

    ('ci 15850 (red 7)',
     ['ci 15850', 'red 7', 'red 6',
      'd&c red no. 7', 'd&c red no. 6']),

    ('ci 45410 (red 28)',
     ['ci 45410', 'red 28', 'd&c red no. 28']),

    ('ci 14700 (red 4)',
     ['ci 14700', 'red 4', 'fd&c red no. 4']),

    ('ci 17200 (red 33)',
     ['ci 17200', 'red 33', 'd&c red no. 33']),

    ('ci 60730 (ext violet 2)',
     ['ci 60730', 'ext violet 2', 'ext. violet 2']),

    ('ci 77266 (carbon black)',
     ['ci 77266', 'carbon black']),

    ('ci 77288 (chromium oxide green)',
     ['ci 77288', 'chromium oxide green',
      'chromium oxide greens']),

    ('ci 77289 (chromium hydroxide green)',
     ['ci 77289', 'chromium hydroxide green']),

    ('synthetic fluorphlogopite',
     ['synthetic fluorphlogopite']),
]


BOTANICAL_WATER_INDICATORS = [
    'leaf water', 'flower water', 'root water',
    'stem water', 'fruit water', 'seed water',
    'petal water', 'bark water', 'herb water',
    'plant water', 'twig water', 'blossom water',
]

GLYCERIN_EXCLUSIONS = [
    'capryloyl glycerin', 'glycerin/sebacic',
    'polyglycerin', 'diglycerin',
    'glyceryl', 'polyglycerol',
    'ethyl hexyl glycerin',
    'ethylhexylglycerin',
]


def apply_canonical_rules(ingredient, rules):
    ing_lower = ingredient.lower()

    is_botanical_water = any(
        bw in ing_lower for bw in BOTANICAL_WATER_INDICATORS
    )
    is_glycerin_compound = any(
        ge in ing_lower for ge in GLYCERIN_EXCLUSIONS
    )

    for canonical, patterns in rules:
        for pattern in patterns:
            pattern_lower = pattern.lower()

            if canonical == 'water' and is_botanical_water:
                continue
            if canonical == 'glycerin' and is_glycerin_compound:
                continue

            idx = ing_lower.find(pattern_lower)
            if idx == -1:
                continue

            before    = ing_lower[idx - 1] if idx > 0 else ' '
            after_idx = idx + len(pattern_lower)
            after     = (ing_lower[after_idx]
                         if after_idx < len(ing_lower) else ' ')

            if before.isalpha() or after.isalpha():
                continue

            return canonical
    return None


canonical_mapping = {}
manually_mapped   = 0
seen_normalized   = {}

for orig_ing, norm_ing in zip(original_names, top_ingredients):
    if norm_ing in seen_normalized:
        canonical_mapping[orig_ing] = seen_normalized[norm_ing]
    else:
        result = apply_canonical_rules(norm_ing, canonical_rules)
        canonical = result if result else norm_ing
        seen_normalized[norm_ing] = canonical
        canonical_mapping[orig_ing] = canonical
        if result:
            manually_mapped += 1

print(f"  {'Unique normalized ingredients':<40} "
      f"{len(seen_normalized):>10,}")
print(f"  {'Mapped by manual rules':<40} "
      f"{manually_mapped:>10,}")
print(f"  {'Remaining unmapped (self-mapped)':<40} "
      f"{len(seen_normalized) - manually_mapped:>10,}")



STEP 3: Apply Manual Canonical Rules

  Unique normalized ingredients                 1,924
  Mapped by manual rules                          380
  Remaining unmapped (self-mapped)              1,544


In [6]:
# ----------------------------------------------------------
# STEP 4: BUILD CANONICAL GROUPS
# ----------------------------------------------------------

print()
print("STEP 4: Build Canonical Groups")
print()

canonical_groups = {}
for ing, canonical in canonical_mapping.items():
    if canonical not in canonical_groups:
        canonical_groups[canonical] = []
    canonical_groups[canonical].append(ing)

canonical_freq = {
    canonical: sum(freq[ing] for ing in variants)
    for canonical, variants in canonical_groups.items()
}

canonical_sorted = sorted(
    canonical_freq.items(),
    key=lambda x: x[1],
    reverse=True
)

print(f"  {'Total canonical groups':<40} "
      f"{len(canonical_groups):>10,}")
print(f"  {'Groups with multiple variants':<40} "
      f"{sum(1 for g in canonical_groups.values() if len(g) > 1):>10,}")
print()
print(f"  {'Rank':<6} {'Canonical Name':<40} "
      f"{'Variants':>10} {'Mentions':>12}")
print(f"  {'----':<6} {'---------------':<40} "
      f"{'--------':>10} {'--------':>12}")
for i, (canonical, mentions) in \
        enumerate(canonical_sorted[:30], 1):
    variants = len(canonical_groups[canonical])
    print(f"  {i:<6} {canonical:<40} "
          f"{variants:>10,} {mentions:>12,}")



STEP 4: Build Canonical Groups

  Total canonical groups                        1,652
  Groups with multiple variants                    85

  Rank   Canonical Name                             Variants     Mentions
  ----   ---------------                            --------     --------
  1      water                                            25        6,805
  2      dimethicone                                      31        4,444
  3      glycerin                                          5        4,393
  4      phenoxyethanol                                    1        4,266
  5      parfum (fragrance)                               11        4,168
  6      tocopherol                                        4        3,053
  7      titanium dioxide (ci 77891)                      11        2,925
  8      linalool                                          2        2,902
  9      limonene                                          2        2,860
  10     silica                             

In [7]:
# ----------------------------------------------------------
# STEP 5: COVERAGE ANALYSIS
# ----------------------------------------------------------

print()
print("STEP 5: Coverage Analysis After Normalization")
print()

total_mentions = len(all_ingredients)

print(f"  {'Top N Groups':<20} {'Groups':>8} "
      f"{'Unique %':>10} {'Coverage %':>12}")
print(f"  {'-------------':<20} {'------':>8} "
      f"{'--------':>10} {'----------':>12}")

for n in [100, 200, 300, 400, 500]:
    top_n_groups   = canonical_sorted[:n]
    mentions_total = sum(m for _, m in top_n_groups)
    unique_pct     = n / len(canonical_groups) * 100
    coverage_pct   = mentions_total / total_mentions * 100
    print(f"  {n:<20,} {n:>8,} "
          f"{unique_pct:>9.1f}% {coverage_pct:>11.1f}%")



STEP 5: Coverage Analysis After Normalization

  Top N Groups           Groups   Unique %   Coverage %
  -------------          ------   --------   ----------
  100                       100       6.1%        51.1%
  200                       200      12.1%        62.8%
  300                       300      18.2%        68.8%
  400                       400      24.2%        72.9%
  500                       500      30.3%        75.8%


In [8]:
# ----------------------------------------------------------
# STEP 6: EXPORT CLASSIFICATION TEMPLATE
# ----------------------------------------------------------

print()
print("STEP 6: Export Classification Template")
print()

classification_rows = []

for rank, (canonical, total_mentions_val) in \
        enumerate(canonical_sorted, 1):

    variants    = canonical_groups[canonical]
    variant_str = ' | '.join(sorted(variants)[:3])
    if len(variants) > 3:
        variant_str += f' ... (+{len(variants)-3} more)'

    classification_rows.append({
        'rank':             rank,
        'canonical_name':   canonical,
        'total_mentions':   total_mentions_val,
        'variant_count':    len(variants),
        'example_variants': variant_str,
        'risk_level':       '',
        'sub_weight':       '',
        'source':           '',
        'health_concern':   ''
    })

classification_df = pd.DataFrame(classification_rows)

classification_df.to_csv(
    os.path.join(QUALITY_PATH, 'classification_template.csv'),
    index=False
)

print(f"  {'Total rows in template':<40} "
      f"{len(classification_df):>10,}")
print(f"  {'Rows to classify (target)':<40} "
      f"{'top 400':>10}")
print(f"  {'File':<40} classification_template.csv")



STEP 6: Export Classification Template

  Total rows in template                        1,652
  Rows to classify (target)                   top 400
  File                                     classification_template.csv


In [9]:
# ----------------------------------------------------------
# STEP 7: SAVE CANONICAL MAPPING
# ----------------------------------------------------------

print()
print("STEP 7: Save Canonical Mapping")
print()

mapping_rows = []
for ing, canonical in canonical_mapping.items():
    mapping_rows.append({
        'raw_ingredient': ing,
        'canonical_name': canonical,
        'frequency':      freq[ing]
    })

mapping_df = pd.DataFrame(mapping_rows).sort_values(
    'frequency', ascending=False
)

mapping_df.to_csv(
    os.path.join(QUALITY_PATH, 'canonical_mapping.csv'),
    index=False
)

print(f"  {'Metric':<45} {'Value':>10}")
print(f"  {'------':<45} {'-----':>10}")
print(f"  {'Raw ingredients after artifact removal':<45} "
      f"{len(original_names):>10,}")
print(f"  {'Unique normalized ingredients':<45} "
      f"{len(seen_normalized):>10,}")
print(f"  {'Canonical groups after normalization':<45} "
      f"{len(canonical_groups):>10,}")
print(f"  {'Reduction in classification work':<45} "
      f"{len(seen_normalized)-len(canonical_groups):>10,}")
print(f"  {'Parsing artifacts removed':<45} "
      f"{len(artifacts_removed):>10,}")
print(f"  {'Formatting artifacts cleaned':<45} "
      f"{formatting_cleaned:>10,}")
print(f"  {'canonical_mapping.csv saved':<45} "
      f"{len(mapping_df):>10,} rows")



STEP 7: Save Canonical Mapping

  Metric                                             Value
  ------                                             -----
  Raw ingredients after artifact removal             1,991
  Unique normalized ingredients                      1,924
  Canonical groups after normalization               1,652
  Reduction in classification work                     272
  Parsing artifacts removed                              9
  Formatting artifacts cleaned                          79
  canonical_mapping.csv saved                        1,991 rows


In [10]:
# ----------------------------------------------------------
# STEP 8: VALIDATE THE COMPLETED TOP-400 CLASSIFICATION
# ----------------------------------------------------------
existing = pd.read_excel(
    os.path.join(QUALITY_PATH, 'classification_top400_COMPLETE.xlsx')
)
new_template = pd.read_csv(
    os.path.join(QUALITY_PATH, 'classification_template.csv')
)

existing_classified = set(existing.loc[existing['risk_level'].notna(), 'canonical_name'])
new_canonical_names = set(new_template['canonical_name'])

# Are any of your classified names missing from the new template?
missing = existing_classified - new_canonical_names
print(f"Classified ingredients no longer in canonical list: {len(missing)}")
if missing:
    for m in sorted(missing):
        print(f"  - {m}")

# Are there new top-ranked ingredients you haven't classified?
new_top_400 = set(new_template.head(400)['canonical_name'])
unclassified_in_top_400 = new_top_400 - existing_classified
print(f"\nNew top-400 ingredients not in your classification: {len(unclassified_in_top_400)}")
if unclassified_in_top_400 and len(unclassified_in_top_400) < 30:
    for u in sorted(unclassified_in_top_400):
        print(f"  - {u}")


Classified ingredients no longer in canonical list: 0

New top-400 ingredients not in your classification: 0
